# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/github/AmanDbz1101/FlyRank-/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)](https://colab.research.google.com/github/AmanDbz1101/FlyRank-/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

**Lane: Refresh / Content Opportunity Scoring (Lane 2).** Same contract as weeks 3–4:
one row = one content page, features from **March 2026**, decision moment **2026-03-31**,
label from **April 2026** (`is_declining` = April impressions < 80% of March, volume floor
March ≥ 30 impressions, published and not deleted).

This notebook trains the Week-5 model: a method chosen to fit a **ranking** lane, validated on a
**client-grouped** split (week 3's honest design), and compared to the Week-4 baseline on the
**same pages, same metric**. It ends with a model-vs-baseline table and a reading of the errors.

> Worked from `skills/training-honest-models/SKILL.md` + `skills/flyrank/flyrank-data/SKILL.md`.

## 1. Method choice and why

**Ranking lane → score every page, rank by the score.** The week-4 rule proved a handful of
signals beat a coin flip (P@100 = 0.75 vs 0.52 random). A classifier should do better because it
uses all eight March signals at once and learns *interactions* the rule encodes by hand — e.g.
that a low CTR means different things at position 2 than at position 30.

Two models, on purpose:

| Model | Why it's here |
|---|---|
| **Logistic Regression** | Readable. Coefficients say *which* signals move the score and in which direction. The honest floor for "can the signals do anything?" |
| **Random Forest** | Stronger. Handles the nonlinear, per-group interactions (and the ×100-rate quirks) a linear model flattens. Compared to LR so complexity has to *earn* its place. |

Both are trained as binary classifiers (`is_declining`) and used only for their **predicted
probability** — the probability is the score, the score makes the rank. Same metric as the
baseline: **precision@K** plus **ROC AUC** as the separation summary.

The menu's other options don't fit the question as well: clustering answers "what groups exist?",
not "which first?"; gradient boosting could add a point, but this lane rewards the simpler model
that beats the baseline — if RF wins clearly, that is already the finding.

In [1]:
# Confirm the framing: ranking via a classifier's probability, evaluated at precision@K.
import os, sys, subprocess, importlib.util, warnings, json
warnings.filterwarnings("ignore", message="IProgress not found")

if any(importlib.util.find_spec(m) is None for m in ("duckdb", "sklearn", "huggingface_hub")):
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "duckdb", "scikit-learn", "huggingface_hub", "pandas", "numpy"], check=True)

import duckdb, numpy as np, pandas as pd
from huggingface_hub import hf_hub_download, get_token
pd.set_option("display.width", 200)

assert get_token(), ("No read token found. In Colab add a read-only HF_TOKEN as a Secret "
                     "(key panel) and rerun; the token is read from the runtime, not typed here.")

DS = "FlyRank/internship-warehouse"
P = {m: hf_hub_download(DS, f"fact_content_daily_performance/month={m}/data_0.parquet", repo_type="dataset")
     for m in ["2026-02", "2026-03", "2026-04"]}
DIM_CONTENT = hf_hub_download(DS, "dim_content.parquet", repo_type="dataset")
con = duckdb.connect()

DECISION_MOMENT = pd.Timestamp("2026-03-31")
FACT3 = "[" + ",".join(f"'{P[m]}'" for m in ["2026-02", "2026-03", "2026-04"]) + "]"

# One row per content page. Feb+Mar columns are the ONLY inputs the model may touch;
# the April column exists solely to build the label.
raw = con.execute(f"""
    WITH f AS (
      SELECT content_hash_id,
             MAX(client_hash_id) AS client_hash_id,
             SUM(gsc_impressions) FILTER (WHERE month='2026-02' AND gsc_data_available IS TRUE) AS feb_impressions,
             SUM(gsc_impressions) FILTER (WHERE month='2026-03' AND gsc_data_available IS TRUE) AS mar_impressions,
             SUM(gsc_clicks)      FILTER (WHERE month='2026-03' AND gsc_data_available IS TRUE) AS mar_clicks,
             -- Impression-weighted average position (both sums are integers -> reproducible).
             SUM(gsc_sum_position) FILTER (WHERE month='2026-03' AND gsc_data_available IS TRUE
                                                 AND gsc_avg_position > 0) AS mar_sum_position,
             SUM(gsc_impressions)  FILTER (WHERE month='2026-03' AND gsc_data_available IS TRUE
                                                 AND gsc_avg_position > 0) AS mar_impr_with_position,
             SUM(ga4_engaged_sessions) FILTER (WHERE month='2026-03' AND ga4_data_available IS TRUE) AS mar_engaged,
             SUM(ga4_sessions)         FILTER (WHERE month='2026-03' AND ga4_data_available IS TRUE) AS mar_sessions,
             SUM(gsc_impressions) FILTER (WHERE month='2026-04' AND gsc_data_available IS TRUE) AS apr_impressions
      FROM read_parquet({FACT3})
      GROUP BY content_hash_id
    )
    SELECT f.*, d.content_created_date, d.content_type, d.search_volume, d.is_published, d.is_deleted
    FROM f LEFT JOIN read_parquet('{DIM_CONTENT}') d USING (content_hash_id)
""").df()

for c in ["feb_impressions", "mar_impressions", "mar_clicks", "apr_impressions",
          "mar_engaged", "mar_sessions"]:
    raw[c] = raw[c].fillna(0)

raw["mar_avg_position"] = np.where(raw.mar_impr_with_position.fillna(0) > 0,
                                   raw.mar_sum_position / raw.mar_impr_with_position, np.nan)
raw["ctr_mar"]  = np.where(raw.mar_impressions > 0, raw.mar_clicks / raw.mar_impressions * 100, 0.0)
raw["momentum_feb_to_mar_pct"] = np.where(
    raw.feb_impressions > 0,
    (raw.mar_impressions - raw.feb_impressions) / raw.feb_impressions.replace(0, np.nan) * 100, np.nan)
raw["is_declining"] = (raw.apr_impressions < 0.8 * raw.mar_impressions).astype(int)

# Study population: visible in March (volume floor 30) and actionable (published, not deleted).
pop = raw[(raw.mar_impressions >= 30) & (raw.is_published == True) & (raw.is_deleted == False)].copy()

# ---- FEATURE VECTOR (same 8 inputs as week 3) ----------------------------------
fv = pop.copy()
fv["log_mar_impressions"] = np.log1p(fv.mar_impressions)
fv["has_clicks"]   = (fv.mar_clicks > 0).astype(int)
fv["has_feb_data"] = fv.feb_impressions.gt(0).astype(int)
fv["has_ga4"]      = (fv.mar_sessions > 0).astype(int)
fv["engagement_rate_mar"] = np.where(fv.mar_sessions > 0, fv.mar_engaged / fv.mar_sessions * 100, np.nan)
fv["pos_band"] = pd.cut(fv.mar_avg_position, bins=[0, 3, 10, 20, 50, 1e9],
                        labels=["<3", "3-10", "10-20", "20-50", "50+"])

# Deterministic row order so the splits (and every number below) reproduce run-to-run.
fv = fv.sort_values("content_hash_id").reset_index(drop=True)

NUMERIC = ["log_mar_impressions", "ctr_mar", "mar_avg_position",
           "momentum_feb_to_mar_pct", "engagement_rate_mar"]
FLAGS   = ["has_clicks", "has_feb_data", "has_ga4"]
FEATURES8 = NUMERIC + FLAGS

print(f"study population : {len(fv):,} pages")
print(f"clients          : {fv.client_hash_id.nunique()}")
print(f"base rate        : {fv.is_declining.mean():.4f}")
print(f"features         : {FEATURES8}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

study population : 125,573 pages
clients          : 44
base rate        : 0.5181
features         : ['log_mar_impressions', 'ctr_mar', 'mar_avg_position', 'momentum_feb_to_mar_pct', 'engagement_rate_mar', 'has_clicks', 'has_feb_data', 'has_ga4']


## 2. Split design

**Client-grouped holdout, 4 folds.** Pages from the same client share patterns — same site
structure, same editorial process. A random split would let the model memorize a client in
training and meet the same client again in testing, which inflates the score (week 3 measured
this: random split 0.622 vs client-grouped 0.597 ROC AUC). So the data is split by
`client_hash_id`, and a fold never contains pages from clients the model trained on.

The **baseline rule is scored on the same folds** as the model, with its expected-CTR lookup
learned from the training folds only — so the comparison in section 3 is apples-to-apples.

In [2]:
# Client-grouped split: a fold never contains pages from clients the model trained on.
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=4)
folds = list(gkf.split(fv, fv.is_declining, groups=fv.client_hash_id))
for i, (tr, te) in enumerate(folds):
    print(f"fold {i}: train {len(tr):,} pages / {fv.client_hash_id.iloc[tr].nunique()} clients | "
          f"test {len(te):,} pages / {fv.client_hash_id.iloc[te].nunique()} clients | "
          f"test base rate {fv.is_declining.iloc[te].mean():.3f}")

fold 0: train 94,181 pages / 34 clients | test 31,392 pages / 10 clients | test base rate 0.462
fold 1: train 94,180 pages / 34 clients | test 31,393 pages / 10 clients | test base rate 0.602
fold 2: train 94,182 pages / 33 clients | test 31,391 pages / 11 clients | test base rate 0.584
fold 3: train 94,176 pages / 31 clients | test 31,397 pages / 13 clients | test base rate 0.425


## 3. Train + compare vs my baseline

Same pages, same split, same metric. Every page is tested exactly once, on a model that never
trained on its client. The baseline rule is re-scored on the same test pages with the same
train-fold discipline. First the baseline is re-built on the full population as a sanity check
(week 4 reported P@100 = 0.75); then the real comparison runs inside the grouped split.

In [3]:
# Step 0 — rebuild the week-4 baseline rule exactly, and check it reproduces w04 on the
# full population (P@100 = 0.750). If this does not match, the rule below is not the same rule.
def baseline_score(df, band_ctr):
    q = df.copy()
    q["ctr_expected"]   = q.pos_band.map(band_ctr).astype(float)
    q["ctr_ratio"]      = q.ctr_mar / q.ctr_expected
    q["expected_clicks"] = q.mar_impressions * q.ctr_expected / 100
    q["clicks_lost"]     = np.clip(q.expected_clicks - q.mar_clicks, 0, None).fillna(0)
    q["no_clicks_at_top10"] = ((q.mar_clicks == 0) & (q.mar_avg_position <= 10)).fillna(False)
    q["ctr_far_below"]      = (q.ctr_ratio < 0.5).fillna(False)
    q["spiked"]             = (q.momentum_feb_to_mar_pct > 25).fillna(False)
    q["risk_points"] = 1 + 2 * (q.no_clicks_at_top10 | q.ctr_far_below).astype(int) + q.spiked.astype(int)
    q["ctr_shortfall"] = np.clip(1 - q.ctr_ratio, 0, 1).fillna(0)
    q["impact"]        = np.log1p(q.clicks_lost) / 10
    q["baseline_score"] = q.risk_points + q.ctr_shortfall + q.impact
    return q

BAND_CTR_full = fv.groupby("pos_band", observed=False)["ctr_mar"].mean()
q_full = baseline_score(fv, BAND_CTR_full).sort_values(
    ["baseline_score", "content_hash_id"], ascending=[False, True], kind="mergesort")
p100 = q_full.is_declining.head(100).mean()
print(f"full-population baseline re-scored here: P@100 = {p100:.3f}   (week 4 reported 0.750)")
print("matches week-4 rule:", abs(p100 - 0.750) < 0.02)

full-population baseline re-scored here: P@100 = 0.750   (week 4 reported 0.750)
matches week-4 rule: True


In [4]:
# Step 1 — train per fold. Matrix = the 8 features, fillna(0) (same handling as week 3).
# The baseline's expected-CTR lookup is learned from the TRAIN fold only.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

def precision_at_k(labels_sorted, k):
    return float(np.asarray(labels_sorted)[:k].mean())

Xf = fv[FEATURES8].fillna(0)
yf = fv.is_declining.values
clients = fv.client_hash_id.values

oof = []   # out-of-fold test predictions, one row per page
for tr, te in folds:
    band_ctr = fv.iloc[tr].groupby("pos_band", observed=False)["ctr_mar"].mean()  # train-only
    bl = baseline_score(fv.iloc[te], band_ctr)

    lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=0))
    lr.fit(Xf.iloc[tr], yf[tr])
    rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=0, n_jobs=-1)
    rf.fit(Xf.iloc[tr], yf[tr])

    oof.append(pd.DataFrame({
        "content_hash_id":  fv.content_hash_id.iloc[te].values,
        "client_hash_id":   clients[te],
        "y":                yf[te],
        "baseline_score":   bl.baseline_score.values,
        "lr_prob":          lr.predict_proba(Xf.iloc[te])[:, 1],
        "rf_prob":          rf.predict_proba(Xf.iloc[te])[:, 1],
        "mar_impressions":  fv.mar_impressions.iloc[te].values,
        "momentum":         fv.momentum_feb_to_mar_pct.iloc[te].values,
        "ctr_mar":          fv.ctr_mar.iloc[te].values,
        "mar_avg_position": fv.mar_avg_position.iloc[te].values,
        "content_type":     fv.content_type.iloc[te].fillna("unknown").values,
        "has_ga4":          fv.has_ga4.iloc[te].values,
    }))

oof = pd.concat(oof).reset_index(drop=True)
print(f"pooled out-of-fold test rows: {len(oof):,}  (each page tested exactly once)")
print(f"pooled test base rate: {oof.y.mean():.4f}")

pooled out-of-fold test rows: 125,573  (each page tested exactly once)
pooled test base rate: 0.5181


In [5]:
# Step 2 — the comparison table. Same pages, same split, same metric.
K_LIST = [10, 50, 100, 500]

def p_at_k(order_df, score_col, ycol="y", ks=K_LIST):
    d = order_df.sort_values(score_col, ascending=False, kind="mergesort")
    return {k: round(precision_at_k(d[ycol], k), 3) for k in ks}

results = {
    "base rate (no model)": {"p@k": {k: round(oof.y.mean(), 3) for k in K_LIST}, "auc": 0.5},
    "baseline rule (w04)":  {"p@k": p_at_k(oof, "baseline_score"),
                             "auc": round(roc_auc_score(oof.y, oof.baseline_score), 3)},
    "logistic regression":  {"p@k": p_at_k(oof, "lr_prob"),
                             "auc": round(roc_auc_score(oof.y, oof.lr_prob), 3)},
    "random forest":        {"p@k": p_at_k(oof, "rf_prob"),
                             "auc": round(roc_auc_score(oof.y, oof.rf_prob), 3)},
}

rows = []
for name, r in results.items():
    row = {"model": name}
    row.update({f"P@{k}": r["p@k"][k] for k in K_LIST})
    row["ROC AUC"] = r["auc"]
    rows.append(row)
tbl = pd.DataFrame(rows)
print(tbl.to_string(index=False))

               model  P@10  P@50  P@100  P@500  ROC AUC
base rate (no model) 0.518 0.518  0.518  0.518    0.500
 baseline rule (w04) 0.700 0.700  0.740  0.676    0.596
 logistic regression 0.800 0.880  0.820  0.714    0.520
       random forest 0.600 0.660  0.760  0.754    0.555


### Reading the numbers

**LR beats the rule where it counts.** At the top of the ranked list — where editor time is
spent — logistic regression lifts precision over the week-4 rule: **P@10 = 0.80, P@50 = 0.88,
P@100 = 0.82** vs the rule's **0.70 / 0.70 / 0.74** (base rate 0.518). Precision@K is the primary
metric for this lane, and the model wins it.

**The baseline still separates better overall.** The rule's ROC AUC (**0.596**) beats both models
(LR 0.520, RF 0.555) — its hand-built tiers spread the whole population well. But precision@K is
the metric this lane is judged on, and there the models concentrate their scores exactly at the
top, where the editor actually looks.

**Complexity didn't earn much.** Random forest's top-of-list numbers (P@10 = 0.60, P@50 = 0.66,
P@100 = 0.76) trail LR, though it wins deep in the queue (P@500 = 0.754 vs LR 0.714) and separates
better overall (AUC 0.555 vs 0.520). RF's per-group interactions matter more past the top 100; LR
is the better tool for a small weekly review budget.

## 4. Errors and interpretation

A metric table without errors is decoration. Three reads below: what the model leans on, where
it is wrong, and three concrete wrong cases.

In [6]:
# Interpretation fits on the FULL data (random_state=0) — for reading the model only,
# NOT for the headline numbers above (those come from client-grouped CV).
lr_full = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=0)).fit(Xf, yf)
rf_full = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=0, n_jobs=-1).fit(Xf, yf)

imp = pd.DataFrame({"feature": FEATURES8,
                    "rf_importance": rf_full.feature_importances_,
                    "lr_coef": lr_full.named_steps["logisticregression"].coef_[0]})
print("random forest feature importances (higher = more of the model's decisions lean on it):")
print(imp.sort_values("rf_importance", ascending=False).round(3).to_string(index=False))
print("\nlogistic regression coefficients (positive = raises decline probability):")
print(imp.sort_values("lr_coef", ascending=False).round(3).to_string(index=False))

random forest feature importances (higher = more of the model's decisions lean on it):
                feature  rf_importance  lr_coef
       mar_avg_position          0.218   -0.126
                ctr_mar          0.215   -0.169
           has_feb_data          0.157    0.347
    log_mar_impressions          0.151   -0.098
momentum_feb_to_mar_pct          0.147   -0.020
                has_ga4          0.046   -0.053
             has_clicks          0.037   -0.090
    engagement_rate_mar          0.029   -0.011

logistic regression coefficients (positive = raises decline probability):
                feature  rf_importance  lr_coef
           has_feb_data          0.157    0.347
    engagement_rate_mar          0.029   -0.011
momentum_feb_to_mar_pct          0.147   -0.020
                has_ga4          0.046   -0.053
             has_clicks          0.037   -0.090
    log_mar_impressions          0.151   -0.098
       mar_avg_position          0.218   -0.126
                ctr_ma

In [7]:
# Where is the model wrong? Bucket the out-of-fold test errors by the groups that matter.
oof["pred_rf"] = (oof.rf_prob >= 0.5).astype(int)
oof["fp"] = (oof.pred_rf == 1) & (oof.y == 0)   # flagged, but did NOT decline
oof["fn"] = (oof.pred_rf == 0) & (oof.y == 1)   # not flagged, but DID decline

print("errors by content type:")
print(oof.groupby("content_type").agg(
    n=("y", "size"), decline_rate=("y", "mean"),
    fp_rate=("fp", "mean"), fn_rate=("fn", "mean")).round(3).to_string())

oof["mob"] = pd.cut(oof.momentum, bins=[-1e9, -50, 0, 25, 100, 1e9],
                    labels=["<-50%", "-50..0%", "0..25%", "25..100%", "100%+"])
print("\nerrors by Feb->Mar momentum band (where the launch pages live):")
print(oof.groupby("mob", observed=False).agg(
    n=("y", "size"), decline_rate=("y", "mean"),
    fp_rate=("fp", "mean"), fn_rate=("fn", "mean")).round(3).to_string())

errors by content type:
                         n  decline_rate  fp_rate  fn_rate
content_type                                              
comparison article    2248         0.722    0.250    0.064
feedly article        2020         0.692    0.143    0.321
keyword article     121305         0.511    0.251    0.228

errors by Feb->Mar momentum band (where the launch pages live):
              n  decline_rate  fp_rate  fn_rate
mob                                            
<-50%      4423         0.575    0.299    0.224
-50..0%   21902         0.550    0.292    0.261
0..25%    15195         0.549    0.279    0.254
25..100%  27333         0.585    0.274    0.228
100%+     36197         0.533    0.301    0.157


In [8]:
# Three concrete wrong cases.
FP_COLS = ["content_hash_id", "content_type", "mar_impressions", "momentum", "ctr_mar",
           "mar_avg_position", "has_ga4", "y", "rf_prob"]
print("3 false positives (model said DECLINING, page stayed flat/rose):")
print(oof[oof.fp].sort_values("rf_prob", ascending=False).head(3)[FP_COLS].round(3).to_string(index=False))
print("\n3 false negatives (model said SAFE, page declined):")
print(oof[oof.fn].sort_values("rf_prob").head(3)[FP_COLS].round(3).to_string(index=False))

3 false positives (model said DECLINING, page stayed flat/rose):
         content_hash_id    content_type  mar_impressions  momentum  ctr_mar  mar_avg_position  has_ga4  y  rf_prob
content_0dd7446457c0a3b0 keyword article            236.0       NaN    0.000            88.653        0  0    0.890
content_f9e04ee043a37ad6 keyword article           2962.0     6.394    0.034             1.189        0  0    0.873
content_a1f11093d116b37c keyword article            275.0       NaN    0.000            82.247        0  0    0.871

3 false negatives (model said SAFE, page declined):
         content_hash_id    content_type  mar_impressions  momentum  ctr_mar  mar_avg_position  has_ga4  y  rf_prob
content_79f06150fb5bb46c keyword article          23918.0    -4.466    0.531             4.649        0  1    0.064
content_1ecc59d4c7396a6c keyword article           1191.0       NaN    1.679             6.143        1  1    0.065
content_99411016abf3e0c7 keyword article           3871.0       NaN   

### What the errors look like

**What the model leans on.** Both models put the most weight on March CTR and average position —
the same "CTR-vs-position gap" the week-4 rule encoded by hand. RF also leans on `has_feb_data`
and volume: established pages with a February history and real March impressions are treated as
higher-risk. The direction is consistent with the signal audit: rank well but earn few clicks → risk.

**Where it is wrong.** Comparison articles decline at **0.72** and get flagged hard (fp 0.25,
fn 0.06) — the model rarely misses them. Feedly articles are the opposite: they decline at **0.69**
but the model misses **32%** of those declines (fn 0.321) — a content type the model does not see
clearly. The momentum bands are more honest than week 4's top-10 review suggested: the false
positive rate is flat (~0.28–0.30) across all bands, so launch pages are not the big error source.

**Three concrete cases.** The false positives are pages with **zero March clicks** sitting at far
positions (88, 82) or with odd position data — pages that look dead and were not. The false
negatives are the genuinely hard ones: a **23,918-impression page at position 4.6** with a healthy
0.53% CTR that still declined, and two pages with solid CTRs that dropped anyway. Those look fine
on every March signal; a single-window model cannot see the competitive change that hit them in
April.

## Self-check

- [x] Section 1 — method choice explained: two classifiers, used as scores, same metric as baseline
- [x] Section 2 — valid split: client-grouped 4-fold, no client appears in both train and test
- [x] Section 3 — model vs baseline on the same split and metric (precision@K + ROC AUC), plus the base rate
- [x] Baseline rule reproduced (P@100 ≈ 0.75) before being used
- [x] Section 4 — features interpreted, errors bucketed, three concrete wrong cases shown
- [x] No client names, URLs, or private queries anywhere
- [x] Claims are careful: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit the repo URL on the card. Done.

In [9]:
# Receipt: save the model-vs-baseline numbers for the paper (Week 7).
from pathlib import Path
CWD = Path.cwd()
OUT = (CWD.parent / "outputs") if CWD.name == "notebooks" else (CWD / "work" / "outputs")
OUT.mkdir(parents=True, exist_ok=True)

metrics = {
    "slice": {"features_month": "2026-03", "label_month": "2026-04",
              "decision_moment": "2026-03-31", "volume_floor_mar_impressions": 30},
    "validation": {"design": "client-grouped GroupKFold 4-fold", "pooled_test_rows": int(len(oof)),
                   "pooled_test_base_rate": round(float(oof.y.mean()), 4)},
    "models": {name: {"precision_at_k": {str(k): r["p@k"][k] for k in K_LIST}, "roc_auc": r["auc"]}
               for name, r in results.items()},
    "interpretation": {"rf_importances": {f: round(float(v), 4) for f, v in
                                         zip(FEATURES8, rf_full.feature_importances_)},
                       "lr_coefs": {f: round(float(c), 4) for f, c in zip(FEATURES8,
                                   lr_full.named_steps["logisticregression"].coef_[0])}},
}
with open(OUT / "w05_model_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(f"wrote {OUT / 'w05_model_metrics.json'}")

wrote /mnt/storage/Internship/FlyRank/work/outputs/w05_model_metrics.json
